# Liberata Portfolio Metrics — Demo

## 0. Imports

In [46]:
import sys, os, json
sys.path.insert(0, os.path.join('..', 'src'))

from datetime import date, timedelta
from pathlib import Path
import numpy as np
from scipy import sparse

from data_loading import load_data

from liberata_metrics.generators import generate_capital_time_series
from liberata_metrics.metrics import (
    academic_capital,
    allocation_weights,
    portfolio_hhi,
    portfolio_gini,
    portfolio_normalized_entropy,
    get_per_manuscript_cap,
    query_portfolio_mix,
    role_based_proportional_loss,
    get_returns,
    get_expected_returns,
    get_volatility,
    get_sharpe_ratio,
    get_arc,
    get_diversification_ratio,
    get_risk_asymmetry,
    get_proportional_split,
    compute_fair_marketprice,
    compute_risk_premiums,
)

print('Imports OK')

Imports OK


## 1. Load Data


In [47]:
BASE_DIR = Path('output/m800_c1200_20260213_154126')

alg_data = load_data(str(BASE_DIR))

capital               = alg_data.capital
contributor_index_map = alg_data.contributor_index_map
manuscript_index_map  = alg_data.manuscript_index_map
primary_memberships   = alg_data.primary_tag_memberships
topic_index_map       = alg_data.topic_index_map
authors_slice         = alg_data.authors_slice
reviewers_slice       = alg_data.reviewers_slice
replicators_slice     = alg_data.replicators_slice
retractions_capital   = alg_data.retractions_capital

# Also load references + shares + upload_dates (needed for time series)
references = sparse.load_npz(str(BASE_DIR / 'references_coo.npz'))
shares     = sparse.load_npz(str(BASE_DIR / 'shares_coo.npz'))
with open(BASE_DIR / 'upload_dates.json') as f:
    upload_dates = {k: date.fromisoformat(v) for k, v in json.load(f).items()}

M = capital.shape[0]
C = len(contributor_index_map)
contributor_ids = list(contributor_index_map.keys())
topic_names     = list(topic_index_map.keys())
START = min(upload_dates.values())
END   = max(upload_dates.values())

print(f'Capital matrix    : {capital.shape}  nnz={capital.nnz}')
print(f'References matrix : {references.shape}  nnz={references.nnz}')
print(f'Manuscripts       : {M}')
print(f'Contributors      : {C}')
print(f'Topics            : {len(topic_index_map)}')
print(f'Date range        : {START} → {END}')
print()

Capital matrix    : (800, 4400)  nnz=7635
References matrix : (800, 800)  nnz=134612
Manuscripts       : 800
Contributors      : 1200
Topics            : 80
Date range        : 2020-01-03 → 2024-12-24



In [44]:
# Use all contributors so capital is spread across all 800 manuscripts — gives meaningful metrics
portfolio_subset = contributor_index_map

# Find a contributor who has capital in all three roles (author + reviewer + replicator)
capital_csc = capital.tocsc()
multi_role_id = None
for cid, idx in contributor_index_map.items():
    has_author     = capital_csc[:, M +       idx].nnz > 0
    has_reviewer   = capital_csc[:, M + C   + idx].nnz > 0
    has_replicator = capital_csc[:, M + 2*C + idx].nnz > 0
    if has_author and has_reviewer and has_replicator:
        multi_role_id = cid
        break

single_contributor = {multi_role_id: contributor_index_map[multi_role_id]}
print(f'Portfolio subset   : all {len(portfolio_subset)} contributors')
print(f'Single contributor : {multi_role_id} (author + reviewer + replicator capital)')

Portfolio subset   : all 1200 contributors
Single contributor : 148f0a3f-4764-4f5b-8657-17bb7ecb9b3c (author + reviewer + replicator capital)


## 2. Core Capital & Allocation Metrics



In [48]:
total_cap = academic_capital(capital, portfolio_subset)
print(f'Total academic capital (30-contributor portfolio): {total_cap:.6f}')

weights = allocation_weights(capital, portfolio_subset)
print(f'\nAllocation weights across {len(weights)} manuscripts:')
top5 = sorted(weights.items(), key=lambda kv: kv[1], reverse=True)[:5]
for ms_idx, w in top5:
    print(f'  manuscript row {ms_idx}: {w:.4f}')

per_ms = get_per_manuscript_cap(capital, portfolio_subset)
print(f'\nPer-manuscript capital submatrix shape: {per_ms.shape}')

Total academic capital (30-contributor portfolio): 701.460651

Allocation weights across 631 manuscripts:
  manuscript row 11: 0.0155
  manuscript row 16: 0.0145
  manuscript row 17: 0.0142
  manuscript row 66: 0.0140
  manuscript row 60: 0.0135

Per-manuscript capital submatrix shape: (800, 1200)


## 3. Concentration & Diversity

Three complementary views on how spread (or concentrated) a portfolio is:

| Metric | Range | Low value means… | High value means… |
|--------|-------|-----------------|------------------|
| **HHI** | 0 → 1 | diversified | concentrated in few manuscripts |
| **Gini** | 0 → 1 | equal allocation | highly unequal |
| **Normalized Entropy** | 0 → 1 | all capital in one manuscript | uniform across all |

In [49]:
hhi     = portfolio_hhi(capital, portfolio_subset)
gini    = portfolio_gini(capital, portfolio_subset)
entropy = portfolio_normalized_entropy(capital, portfolio_subset)

print(f'HHI                : {hhi:.4f}   (0=diversified, 1=concentrated)')
print(f'Gini coefficient   : {gini:.4f}   (0=equal, 1=maximally unequal)')
print(f'Normalized entropy : {entropy:.4f}  (0=all in one paper, 1=perfectly uniform)')

HHI                : 0.0052   (0=diversified, 1=concentrated)
Gini coefficient   : 0.6564   (0=equal, 1=maximally unequal)
Normalized entropy : 0.8779  (0=all in one paper, 1=perfectly uniform)


## 4. Portfolio Composition by Role & Topic

`query_portfolio_mix` breaks a portfolio down by role (author / reviewer / replicator)
and then by topic tag within each role.

In [50]:
role_totals, tag_mix_per_role = query_portfolio_mix(
    capital=capital,
    mask_authors=authors_slice,
    mask_reviewers=reviewers_slice,
    mask_replicators=replicators_slice,
    manuscript_memberships=primary_memberships,
    contributor_index_map_subset=single_contributor,
    by='role',
)

role_names = ['Author', 'Reviewer', 'Replicator']
print('Capital by role (single contributor):')
for name, cap in zip(role_names, role_totals):
    print(f'  {name:12s}: {cap:.6f}')

print('\nTop topic for each role:')
for name, tag_mix in zip(role_names, tag_mix_per_role):
    if tag_mix is not None and tag_mix.nnz > 0:
        arr = np.asarray(tag_mix.todense()).ravel()
        top_idx = int(np.argmax(arr))
        print(f'  {name:12s}: {topic_names[top_idx]}')
    else:
        print(f'  {name:12s}: (no capital)')

Capital by role (single contributor):
  Author      : 0.000254
  Reviewer    : 0.097379
  Replicator  : 0.285345

Top topic for each role:
  Author      : Halal products and consumer behavior
  Reviewer    : Educational Research and Pedagogy
  Replicator  : Cultural History and Identity Formation


## 5. Time Series — Returns & Risk

`generate_capital_time_series` produces capital snapshots at regular intervals
from the loaded references and shares, respecting temporal ordering.

In [51]:
ts = generate_capital_time_series(
    references=references,
    shares=shares,
    manuscript_index_map=manuscript_index_map,
    upload_dates=upload_dates,
    start_date=START,
    end_date=END,
    time_step=timedelta(days=90),
)

snapshots  = ts['capital_snapshots']
timestamps = ts['timestamps']

print(f'Time steps     : {len(snapshots)}')
print(f'First snapshot : {timestamps[0]}')
print(f'Last snapshot  : {timestamps[-1]}')
print(f'Snapshot shape : {snapshots[0].shape}')

Time steps     : 21
First snapshot : 2020-01-03
Last snapshot  : 2024-12-07
Snapshot shape : (1, 3601)


In [52]:
returns_simple   = get_returns(snapshots[0], snapshots[-1], portfolio_subset)
returns_expected = get_expected_returns(snapshots, portfolio_subset)

print(f'Simple returns (start → end) : {returns_simple:.6f}')
print(f'Expected returns (mean)      : {returns_expected:.6f}')

Simple returns (start → end) : 0.000000
Expected returns (mean)      : 0.187300


In [53]:
volatility     = get_volatility(snapshots, portfolio_subset)
risk_asymmetry = get_risk_asymmetry(snapshots, portfolio_subset)

print(f'Volatility (std dev of returns)  : {volatility:.6f}')
print(f'Risk asymmetry (skew of returns) : {risk_asymmetry:.6f}')
print('  (positive = upside-skewed, negative = downside-skewed)')

Volatility (std dev of returns)  : 0.195315
Risk asymmetry (skew of returns) : 1.778191
  (positive = upside-skewed, negative = downside-skewed)


In [54]:
sharpe = get_sharpe_ratio(snapshots, portfolio_subset)
arc    = get_arc(snapshots, portfolio_subset)

print(f'Sharpe ratio : {sharpe:.4f}  (higher = better risk-adjusted return)')
print(f'ARC          : {arc:.4f}  (Academic Returns-to-Capital ratio)')

# get_diversification_ratio is skipped here: it assumes all snapshots have the same number of
# rows, but generate_capital_time_series returns variable-row snapshots (only manuscripts
# published up to each time point), causing an IndexError on early snapshots.

Sharpe ratio : 0.9590  (higher = better risk-adjusted return)
ARC          : 0.0001  (Academic Returns-to-Capital ratio)


## 6. Market Metrics — Fair Market Price & Risk Premiums

The **fair market price (FMP)** for a topic is the average capital per manuscript in that topic,
serving as a reference rate for reviewers and replicators.

A **risk premium** is the difference between what a contributor actually received and the FMP —
positive means they were rewarded above the market rate.

In [55]:
reviewer_fmp, replicator_fmp = compute_fair_marketprice(
    capital=capital,
    mask_reviewers=reviewers_slice,
    mask_replicators=replicators_slice,
    manuscript_memberships=primary_memberships,
    contributor_index_map_subset=contributor_index_map,
    num_contributors=C,
)

print(f'Reviewer FMP shape   : {reviewer_fmp.shape}  (topics × contributors)')
print(f'Replicator FMP shape : {replicator_fmp.shape}')

rev_fmp_arr = np.asarray(reviewer_fmp.mean(axis=1)).ravel()
print(f'\nMean reviewer FMP per topic (top 5):')
for i in np.argsort(rev_fmp_arr)[::-1][:5]:
    print(f'  {topic_names[i][:55]:55s}: {rev_fmp_arr[i]:.6f}')

Reviewer FMP shape   : (80, 1)  (topics × contributors)
Replicator FMP shape : (80, 1)

Mean reviewer FMP per topic (top 5):
  Hemophilia Treatment and Research                      : 0.052221
  Educational Research and Pedagogy                      : 0.010820
  Environmental and Industrial Safety                    : 0.009976
  Philosophy and History of Science                      : 0.002076
  Nostalgia and Consumer Behavior                        : 0.001849


## 7. Additional Metrics

- **`get_proportional_split`** — fraction of a portfolio's capital earned via a specific role
- **`role_based_proportional_loss`** — fraction of capital lost to retractions for a role (uses real retraction data from the loaded dataset)

In [56]:
author_split     = get_proportional_split(capital, authors_slice,     portfolio_subset)
reviewer_split   = get_proportional_split(capital, reviewers_slice,   portfolio_subset)
replicator_split = get_proportional_split(capital, replicators_slice, portfolio_subset)

print('Proportional role split (all contributors):')
print(f'  Author      : {author_split:.4f}')
print(f'  Reviewer    : {reviewer_split:.4f}')
print(f'  Replicator  : {replicator_split:.4f}')
print(f'  Sum         : {author_split + reviewer_split + replicator_split:.4f}')

Proportional role split (all contributors):
  Author      : 0.9956
  Reviewer    : 0.0012
  Replicator  : 0.0032
  Sum         : 1.0000


In [57]:
author_loss     = role_based_proportional_loss(capital, retractions_capital, authors_slice,     portfolio_subset)
reviewer_loss   = role_based_proportional_loss(capital, retractions_capital, reviewers_slice,   portfolio_subset)
replicator_loss = role_based_proportional_loss(capital, retractions_capital, replicators_slice, portfolio_subset)

print('Proportional capital loss to retractions:')
print(f'  Author      : {author_loss:.4f}')
print(f'  Reviewer    : {reviewer_loss:.4f}')
print(f'  Replicator  : {replicator_loss:.4f}')

Proportional capital loss to retractions:
  Author      : 0.1162
  Reviewer    : 0.2481
  Replicator  : 0.0690
